# Visual intuition atlas

A companion to the chronological research notes. These plots are deliberately small and synthetic. Their purpose is to make the mechanics behind the research questions visible: what changes when we vary phase, radius, recurrence depth, checkpoint selection, context length, or recoverable signal strength?

Unless a plot is explicitly labeled **historical result**, its numbers are illustrative and should not be read as experimental evidence from the private research program.


## 1. Representation can change coordinates without losing information

Related milestone: [001 — What if text were a signal?](001_text_as_signal.ipynb)

A Fourier transform is invertible, but it changes which structure is easy to see. The same waveform can be inspected in the original coordinate basis or by frequency.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 1, 256, endpoint=False)
x = np.sin(2*np.pi*5*t) + 0.45*np.sin(2*np.pi*17*t)
freq = np.fft.rfftfreq(len(t), d=t[1]-t[0])
mag = np.abs(np.fft.rfft(x))

plt.figure(figsize=(7,3))
plt.plot(t, x)
plt.xlabel('Position')
plt.ylabel('Signal value')
plt.title('The same information in the original coordinates')
plt.show()

plt.figure(figsize=(7,3))
plt.plot(freq, mag)
plt.xlim(0, 30)
plt.xlabel('Frequency')
plt.ylabel('Magnitude')
plt.title('A coordinate change exposes different structure')
plt.show()


> **Sticky note — coordinate system:** An invertible coordinate change can preserve all information while changing how complicated a downstream operation must be to use it.

The visible peaks do not prove a spectral language model is better. They illustrate why “information is preserved” and “information is easy for this consumer to use” are different claims.


## 2. Phase-aware similarity responds directly to phase difference

Related milestones: [003](003_coordinates_and_matched_operations.ipynb) and [004](004_isolating_phase_attention.ipynb).

For unit complex numbers, normalized real-Hermitian similarity reduces to `cos(Δθ)`. Varying relative phase therefore changes similarity in a predictable circular way.


In [ ]:
delta = np.linspace(-np.pi, np.pi, 400)
similarity = np.cos(delta)
plt.figure(figsize=(7,3))
plt.plot(delta, similarity)
plt.axhline(0, linewidth=1)
plt.xlabel('Phase difference Δθ (radians)')
plt.ylabel('Normalized real-Hermitian similarity')
plt.title('Phase-aware similarity is circular')
plt.show()


Aligned phases score positively, quadrature is near zero, and opposition scores negatively.

**Understanding question:** Why is this more structurally meaningful for phase-valued features than comparing their raw Cartesian coordinates with an arbitrary learned rule?


## 3. A frozen endpoint can reverse a trajectory story

Related milestones: [005](005_unit_hypersphere_anomaly.ipynb) and [006](006_endpoint_can_mislead.ipynb). The curves below are illustrative.


In [ ]:
steps = np.array([0,16,32,64,96,128,192,288])
shared = np.array([5.4,3.7,2.95,3.02,3.18,3.38,3.55,3.72])
unit = np.array([5.5,4.5,4.05,3.85,3.72,3.60,3.42,3.30])
plt.figure(figsize=(7,3))
plt.plot(steps, shared, marker='o', label='Shared recurrent control')
plt.plot(steps, unit, marker='o', label='Unit-hypersphere model')
plt.xlabel('Training update')
plt.ylabel('Illustrative validation NLL')
plt.title('Ranking depends on the checkpoint-selection question')
plt.legend()
plt.show()


A late fixed endpoint asks “which model is better here?” A best-observed checkpoint asks “what was the best state encountered?” A frozen stopping policy asks “what would a fair procedure have selected without hindsight?” Those estimands can rank the same trajectories differently.


## 4. Normalization removes radial motion and rescales tangent motion

Related milestone: [007 — Derive before training](007_derive_before_training.ipynb). For `N(x)=x/||x||`, radial first-order gain is `0` and tangent gain is `1/||x||`.


In [ ]:
r = np.linspace(0.2, 4.0, 300)
plt.figure(figsize=(7,3))
plt.plot(r, 1/r, label='Tangent gain 1/r')
plt.plot(r, np.zeros_like(r), label='Radial gain 0')
plt.axvline(1.0, linestyle='--', linewidth=1)
plt.xlabel('Input radius ||x||')
plt.ylabel('First-order gain')
plt.title('Normalization is exactly anisotropic')
plt.legend()
plt.show()


At unit radius, tangent perturbations are preserved to first order. At larger radius they are attenuated; near zero they are strongly amplified, which is one reason the nonzero-domain assumption matters.

✓ **Formal checkpoint:** the exact derivative statement is machine-checked in the public theorem library.


## 5. Small per-step differences compound across recurrence

Related milestone: [008 — What does the sphere actually do?](008_frozen_mechanism_tests.ipynb). For a scalar mode with per-step gain `g`, a T-step recurrence carries the perturbation with gain `g^T`.


In [ ]:
depth = np.arange(0, 21)
plt.figure(figsize=(7,3))
for g in [0.98,0.9,0.7]:
    plt.plot(depth, g**depth, marker='o', label=f'g={g}')
plt.xlabel('Recurrent depth')
plt.ylabel('Perturbation gain')
plt.title('Mild one-step contraction can become strong finite-horizon contraction')
plt.legend()
plt.show()


In real networks the singular directions can rotate, so the actual Jacobian product—not independent per-step singular values—is the relevant finite-horizon object.


## 6. Radius and angle are coupled in ordinary Euclidean polar geometry

Related milestone: [011 — Geometry should follow invariance](011_geometry_from_invariance.ipynb). For fixed angle `θ`, raw inner product is `r_q r_k cos(θ)`.


In [ ]:
rq = np.linspace(0.5,3.0,80); rk = np.linspace(0.5,3.0,80)
RQ,RK = np.meshgrid(rq,rk)
S = RQ*RK*np.cos(np.pi/3)
plt.figure(figsize=(6,4))
plt.imshow(S, origin='lower', aspect='auto', extent=[rq.min(),rq.max(),rk.min(),rk.max()])
plt.colorbar(label='Raw inner product')
plt.xlabel('Query radius'); plt.ylabel('Key radius')
plt.title('At fixed angle, Euclidean similarity still changes with radius')
plt.show()


If semantic similarity is supposed to be invariant to radius, this coupling is a design mismatch. If radius should modulate similarity, the same coupling may be intentional. Geometry should follow the invariance the task requires.


## 7. Longer contexts can increase specificity while reducing repeated support

Related milestone: [012 — Natural-source task distinctions](012_natural_source_distinctions.ipynb). The following counts are illustrative, not historical benchmark counts.


In [ ]:
L = np.array([8,16,32,64,128])
repeated = np.array([1200,760,410,125,18])
branches = np.array([310,245,180,62,9])
plt.figure(figsize=(7,3))
plt.plot(L,repeated,marker='o',label='Repeated exact contexts')
plt.plot(L,branches,marker='o',label='Contexts with distinct continuations')
plt.xscale('log',base=2)
plt.xlabel('Exact context length L (bytes)')
plt.ylabel('Illustrative support count')
plt.title('Specificity and repeated support trade off')
plt.legend()
plt.show()


The actual benchmark qualified at `L=32`; the plot explains why context length must be frozen before favorable model outcomes are inspected.


## 8. Probe recoverability changes continuously with signal strength

Related milestone: [013 — Accessible does not imply used](013_accessible_not_used.ipynb). This sweep changes only the linearly accessible class signal embedded in noisy synthetic states.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
rng=np.random.default_rng(12); strengths=np.linspace(0,1.5,13); scores=[]
for strength in strengths:
    n,d=3000,24; y=rng.integers(0,2,n); h=rng.normal(size=(n,d))
    v=rng.normal(size=d); v/=np.linalg.norm(v)
    h+=(2*y[:,None]-1)*strength*v
    Xt,Xe,yt,ye=train_test_split(h,y,test_size=.3,random_state=5,stratify=y)
    scores.append(LogisticRegression(max_iter=1000).fit(Xt,yt).score(Xe,ye))
plt.figure(figsize=(7,3))
plt.plot(strengths,scores,marker='o')
plt.axhline(.5,linestyle='--',linewidth=1)
plt.xlabel('Injected linearly accessible signal strength')
plt.ylabel('Held-out affine-probe accuracy')
plt.title('Accessible information is not a binary property')
plt.show()


A strong probe result establishes recoverability for the tested family, not native causal use. A weak result can reflect low signal, limited data, probe-family mismatch, or protocol/optimization failure.

## How to use this atlas

Change one parameter at a time and predict the qualitative effect before rerunning the cell. The goal is operational intuition, not formula memorization. Ask repeatedly: what is fixed, what is varied, what alternative explanation gives the same picture, and what result would change the next experiment?
